## ACUTE MYELOID LUEKANEMIA PHENOTYPE FILE 

In [ ]:
# Load packages 
from sklearn.preprocessing import StandardScaler
from functools import reduce
import pandas as pd 
import os 

In [ ]:
# Workspace bucket 
my_bucket = os.getenv("WORKSPACE_BUCKET")

In [ ]:
# Make directory for aml phenotypes files 
!rm -rf aml_phenotype_files
!mkdir -p aml_phenotype_files

In [ ]:
# Assign sex variable 
sex['sex'] = sex['dragen_sex_ploidy'].apply(lambda x: 1 if x == "XX" else 0 if x == "XY" else 2)
sex['sex'] = sex['sex'].astype("int")

In [ ]:
import pandas
import os

# This query represents dataset "aml_cohort" for domain "person" and was generated for All of Us Controlled Tier Dataset v8
dataset_39009800_person_sql = """
    SELECT
        person.person_id 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.person` person   
    WHERE
        person.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                criteria.person_id 
            FROM
                (SELECT
                    DISTINCT person_id, entry_date, concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                WHERE
                    (concept_id IN(SELECT
                        DISTINCT c.concept_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                    JOIN
                        (SELECT
                            CAST(cr.id as string) AS id       
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                        WHERE
                            concept_id IN (140352)       
                            AND full_text LIKE '%_rank1]%'      ) a 
                            ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                            OR c.path LIKE CONCAT('%.', a.id) 
                            OR c.path LIKE CONCAT(a.id, '.%') 
                            OR c.path = a.id) 
                    WHERE
                        is_standard = 1 
                        AND is_selectable = 1) 
                    AND is_standard = 1 )) criteria 
            UNION
            DISTINCT SELECT
                criteria.person_id 
            FROM
                (SELECT
                    DISTINCT person_id, entry_date, concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                WHERE
                    (concept_id IN (4304199) 
                    AND is_standard = 1 )) criteria ) )"""

aml_person_df = pandas.read_gbq(
    dataset_39009800_person_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

In [ ]:
import pandas
import os

# This query represents dataset "electronic_health_records" for domain "person" and was generated for All of Us Controlled Tier Dataset v8
dataset_72216074_person_sql = """
    SELECT
        person.person_id 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.person` person   
    WHERE
        person.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
            WHERE
                has_ehr_data = 1 ) )"""

ehr_person_df = pandas.read_gbq(
    dataset_72216074_person_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

In [ ]:
import pandas
import os

# This query represents dataset "zip_code_socioeconmic_cohort" for domain "zip_code_socioeconomic" and was generated for All of Us Controlled Tier Dataset v8
dataset_01113255_zip_code_socioeconomic_sql = """
    SELECT
        observation.person_id,
        observation.observation_datetime,
        zip_code.zip3_as_string as zip_code,
        zip_code.fraction_assisted_income as assisted_income,
        zip_code.fraction_high_school_edu as high_school_education,
        zip_code.median_income,
        zip_code.fraction_no_health_ins as no_health_insurance,
        zip_code.fraction_poverty as poverty,
        zip_code.fraction_vacant_housing as vacant_housing,
        zip_code.deprivation_index,
        zip_code.acs as american_community_survey_year 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.zip3_ses_map` zip_code 
    JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.observation` observation 
            ON CAST(SUBSTR(observation.value_as_string, 0, STRPOS(observation.value_as_string, '*') - 1) AS INT64) = zip_code.zip3 
            AND observation_source_concept_id = 1585250 
            AND observation.value_as_string NOT LIKE 'Res%'"""

zip_code_socioeconomic_df = pandas.read_gbq(
    dataset_01113255_zip_code_socioeconomic_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

zip_code_socioeconomic_df.head(5)

In [ ]:
# Combine list of quality control information and demographic information 
phenotype_file = zip_code_socioeconomic_df[zip_code_socioeconomic_df["person_id"].isin(ehr_person_df["person_id"])]

In [ ]:
# Assign AML phenotype to individuals 
phenotype_file['aml_phenotype'] = phenotype_file['person_id'].isin(aml_person_df['person_id']).astype(int)

In [ ]:
# Format phenotype file  
phenotype_file = (phenotype_file[['aml_phenotype',
            'deprivation_index', 'assisted_income', 
            'high_school_education','median_income',
            'no_health_insurance', 'poverty',
            'vacant_housing', 
            'person_id']]
)

In [ ]:
# Identify columns that start with "PC" or "age"
cols_to_standardize = [col for col in phenotype_file.columns if col.startswith("PC") 
                       or col.startswith("age") 
                       or col.startswith("deprivation_index")
                       or col.startswith("assisted_income")
                       or col.startswith("high_school_education")
                       or col.startswith("median_income")
                       or col.startswith("no_health_insurance")
                       or col.startswith("poverty")
                       or col.startswith("vacant_housing")]
# Standardize only the selected columns
scaler = StandardScaler()
phenotype_file[cols_to_standardize] = scaler.fit_transform(phenotype_file[cols_to_standardize])
# Print results of the standarization 
print("============================================")
print("Check variances of standardized covariates:")
print("============================================")
print(phenotype_file.filter(regex="PC|age").var())
print(phenotype_file.describe())
print("============================================")

In [ ]:
# Final check for phenotype 
print("============================================")
print("Check the presence of aml phenotype:")
print("============================================")
print(phenotype_file["aml_phenotype"].value_counts())
print("============================================")

In [ ]:
# Save phenoytpe file
phenotype_file.to_csv("aml_phenotype_deprivation_index.txt", sep ='\t', index=None)